# Single .fcs File Analysis

Full pipeline: load → compensate → transform → gate → cluster → dimensionality reduction.  
Uses Bioconductor packages: flowCore, ggcyto, FlowSOM, openCyto.

## 1. Setup

In [ ]:
library(flowCore)
library(ggcyto)
library(FlowSOM)
library(openCyto)
library(flowWorkspace)
library(ggplot2)
library(uwot)
library(viridis)
library(patchwork)

## 2. Load .fcs file

In [ ]:
# -- Point this to your .fcs file --
fcs_path <- "../data/sample.fcs"
ff <- read.FCS(fcs_path, transformation = FALSE, truncate_max_range = FALSE)

ff
cat("\nEvents:", nrow(ff), "\n")
cat("Parameters:", paste(colnames(ff), collapse = ", "), "\n")

## 3. Inspect metadata

In [ ]:
# Parameter annotations
pData(parameters(ff))

# Keywords of interest
keyword(ff)[grep("SPILL|COMP|DATE|CYT|SRC", names(keyword(ff)), ignore.case = TRUE)]

## 4. Compensation

In [ ]:
# Try to extract spillover matrix from the file
spill <- keyword(ff)$SPILL
if (is.null(spill)) spill <- keyword(ff)$`$SPILLOVER`
if (is.null(spill)) spill <- keyword(ff)$COMP

if (!is.null(spill)) {
    ff_comp <- compensate(ff, spill)
    cat("Compensation applied.\n")
} else {
    ff_comp <- ff
    cat("No spillover matrix found in file — skipping compensation.\n")
    cat("If needed, supply one manually: ff_comp <- compensate(ff, your_matrix)\n")
}

## 5. Transformation

Logicle (biexponential) is standard for fluorescence channels. Scatter channels are typically left untransformed.

In [ ]:
# Identify fluorescence channels (exclude FSC, SSC, Time)
all_channels <- colnames(ff_comp)
scatter_pattern <- "^(FSC|SSC|Time)"
fluor_channels <- grep(scatter_pattern, all_channels, value = TRUE, invert = TRUE)

cat("Fluorescence channels to transform:", paste(fluor_channels, collapse = ", "), "\n")

# Logicle transformation
lgcl <- estimateLogicle(ff_comp, channels = fluor_channels)
ff_trans <- transform(ff_comp, lgcl)

## 6. Scatter plots — raw vs transformed

In [ ]:
# Pick two fluorescence channels for visualization
if (length(fluor_channels) >= 2) {
    ch1 <- fluor_channels[1]
    ch2 <- fluor_channels[2]
} else {
    ch1 <- fluor_channels[1]
    ch2 <- "SSC-A"
}

# Transformed data as data.frame
df_trans <- as.data.frame(exprs(ff_trans))

p1 <- ggplot(df_trans, aes(x = .data[[ch1]], y = .data[[ch2]])) +
    geom_hex(bins = 150) +
    scale_fill_viridis() +
    theme_minimal() +
    labs(title = paste(ch1, "vs", ch2, "(transformed)"))

p2 <- ggplot(df_trans, aes(x = `FSC-A`, y = `SSC-A`)) +
    geom_hex(bins = 150) +
    scale_fill_viridis() +
    theme_minimal() +
    labs(title = "FSC-A vs SSC-A")

p2 + p1

## 7. Manual gating with flowWorkspace

In [ ]:
# Create a GatingSet from the flowFrame
gs <- GatingSet(flowSet(ff_trans))

# Gate 1: Scatter gate (debris removal)
# -- Adjust coordinates to your data --
scatter_gate <- rectangleGate(
    filterId = "nonDebris",
    "FSC-A" = c(50000, 250000),
    "SSC-A" = c(5000, 200000)
)

gs_pop_add(gs, scatter_gate, parent = "root")
recompute(gs)

gs_get_pop_paths(gs)
gs_pop_get_stats(gs)

In [ ]:
# Visualize the scatter gate
ggcyto(gs, aes(x = `FSC-A`, y = `SSC-A`)) +
    geom_hex(bins = 150) +
    geom_gate("nonDebris") +
    geom_stats() +
    scale_fill_viridis() +
    theme_minimal()

## 8. Automated gating with openCyto (optional)

In [ ]:
# Example: mindensity gate on a fluorescence channel
# Finds the valley between negative and positive populations
if (length(fluor_channels) >= 1) {
    tryCatch({
        g <- openCyto::gate_mindensity(
            gs_pop_get_data(gs, "nonDebris")[[1]],
            channel = fluor_channels[1],
            filterId = paste0(fluor_channels[1], "_pos")
        )
        gs_pop_add(gs, g, parent = "nonDebris")
        recompute(gs)
        gs_pop_get_stats(gs)
    }, error = function(e) {
        message("Automated gate skipped: ", conditionMessage(e))
    })
}

## 9. FlowSOM clustering

In [ ]:
# Extract gated population
gated_data <- gs_pop_get_data(gs, "nonDebris")[[1]]

# Run FlowSOM on fluorescence channels
fsom <- FlowSOM(
    gated_data,
    colsToUse = fluor_channels,
    nClus = 10,      # meta-clusters
    xdim = 10,
    ydim = 10,
    seed = 42
)

# Get cluster assignments
meta_clusters <- GetMetaclusters(fsom)
cat("Meta-cluster sizes:\n")
table(meta_clusters)

In [ ]:
# FlowSOM tree
PlotStars(fsom, backgroundValues = GetMetaclusters(fsom))

## 10. UMAP

In [ ]:
mat <- exprs(gated_data)[, fluor_channels]
mat_scaled <- scale(mat)

set.seed(42)
n_sub <- min(nrow(mat_scaled), 10000)
idx <- sample(nrow(mat_scaled), n_sub)

umap_res <- uwot::umap(mat_scaled[idx, ], n_neighbors = 15, min_dist = 0.2)

umap_df <- data.frame(
    UMAP1 = umap_res[, 1],
    UMAP2 = umap_res[, 2],
    cluster = factor(meta_clusters[idx])
)

In [ ]:
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, color = cluster)) +
    geom_point(alpha = 0.3, size = 0.5) +
    theme_minimal() +
    labs(title = "UMAP — FlowSOM meta-clusters")

In [ ]:
# Color by marker intensity
for (ch in fluor_channels) {
    umap_df[[ch]] <- mat[idx, ch]
}

plots <- lapply(fluor_channels, function(ch) {
    ggplot(umap_df, aes(x = UMAP1, y = UMAP2, color = .data[[ch]])) +
        geom_point(alpha = 0.3, size = 0.3) +
        scale_color_viridis(option = "C") +
        theme_minimal() +
        labs(title = ch)
})

wrap_plots(plots)

## 11. Summary statistics

In [ ]:
# Population stats
pop_stats <- gs_pop_get_stats(gs, type = "count")
pop_stats$percent <- pop_stats$count / pop_stats$count[1] * 100
pop_stats

# Cluster MFIs
cluster_mfi <- aggregate(mat[idx, ], by = list(cluster = meta_clusters[idx]), FUN = median)
cluster_mfi